# Danish Agriculture Agency Playground

This notebook is meant as a playground to get familiar with the [Danish Agriculture Agency](https://landbrugsgeodata.fvm.dk/) different datasets.

In [26]:
import ee
import geemap
import os
from dotenv import load_dotenv
import numpy as np
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

# Define map visual parameters
DW_CLASS = {
    'water' : '419bdf', # blue
    'trees' : '397d49', # green
    'grass' : '88b053', # light green
    'flooded_vegetation' : '7a87c6', # greyish blue
    'crops' : 'e49635', # orange
    'shrub_and_scrub' : 'dfc35a', # yellow
    'built' : 'c4281b', # red
    'bare' : 'a59b8f', # grey
    'snow_and_ice' : 'b39fe1', # purple
}
DW_VIZ_PARAMS = {
  'min' : 0,
  'max' : 8,
  'palette' : list(DW_CLASS.values()),
  'bands' : 'label'
}

# B2, B3, B4 are the optical visualization bands, 
# min defines the values that should be mapped to RGB(0) and max the values that should be mapped to RGB(255)
S2_VIZ_PARAMS = {
  'min' : 0,
  'max' : 3000,
  'bands' : ['B4', 'B3', 'B2']
}

# Define Denmark Area
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark'))

YEAR = 20

The datasets coming from DAA come in the format of shapefiles. To use the dataset as Earth Engine Object we first need to upload it to [EE Code Editor](https://code.earthengine.google.com/). Geemap function `shp_to_ee`  would have also been an option, but it requires the CRS of our original dataset to be EPSG:4326, which is not the case. Earth Engine only cares about the .shp, .shx, .dbf and .prj fil extensions. When uploading the DAA datasets is important to clarify that our dataset follows `ISO 88591` and not the standard `UTF-8`. 

Once our dataset is uploaded we can now access it as a `FeatureCollection` object.

In [27]:
dataset = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR))
display(dataset.limit(5))

## Exploring the DAA Dataset

### Markblokke

A Markblok is a continuous geographical area of farmland defined by permanent boundaries in the landscape (like roads, hedges, or streams). Inside a single field block, one or more farmers might have several individual crops (fields). [Reference](https://geodata-info.dk/srv/api/recordsd91b2c99-d9b0-4e6d-b323-20ac80548186#:~:text=Markblokkortet%20er%20et%20digitalt%20markkort,typisk%20permanente%20skel%20i%20landskabet.) The data collected comes from information provided by the farmers which enter the information to apply for EU CAP subsidies, aerial pictures and is reviewed by professional from the agency. 

Our dataset has the following columns:

- **MARKBLOKNR** (String): unique identification number for a specific field block
- **GEOMETRISK** (Float): area without deducting non-eligible area in hectares
- **GB_FRADRAG** (Float): area *not* eligible for standard EU agricultural support (such as large clumps of trees, buildings, or roads) in hectares
- **GB_AREAL** (Float): area eligible for subsidy in hectares
- **MB_TYPE** (String/Enum): type of the field block ([Reference](https://lbst.dk/Media/638467079774681715Brugerguide_marker_og_markblokke.pdf))
    - OMD - Arable land/Rotation land
    - PGR - Permanent grass (grass)
    - PAF - Permanent crops (crops)
    - MIX - Mixed field block
    - VKS - Greenhouses
    - LDP - Sustained/Afforested Woodland
    - ING - No support
    - SOL - Solar Farms

## Total Field Area: How big is the total field area?

We consider the total field area to be the sum of areas of the polygons which define the FeatureCollection. When uploading the dataset to GEE, we noticed a lost in precision. This can be explained by GEE using EPSG: 4326 standard for spatial data, while the dataset provided is originally projected following EPSG:4258. The `GEOMETRISK` field is calculated with the original polygon sizes (this was confirmed  using QGIS desktop). We can quantify the loss in precision by checking the difference between the Feature Collection Area size and the sum of the values in the `GEOMETRISK` column.

As for `GB_AREAL` and `GB_FRADRAG` the only reason I can come up for the difference is the precision lost by rounding every area to two decimal places.


|  | Feature Collection DAA | GEOMETRISK Sum | GB_AREAL Sum | GB_FRADRAG SUM | AREAL + FRADRAG SUM | Percentage Difference 
| --- | --- | --- | --- | --- | --- | --- |
| 2020 | 27778.07 | 27871.08 | 26041.32 | 150.89 | 26192.21 | 0.33 |
| 2021 | 27567.54 | 27660.21 | 26010.36 | 119.22 | 26129.58 | 0.33 |
| 2022 | 27359.32 | 27451.80 | 25977.96 | 115.20 | 26093.16 | 0.33 |
| 2023 | 27018.07 | 27110.70 | 26048.40 | 102.20 | 26150.60 | 0.34 |
| 2024 | 26982.90 | 27075.33 | 26171.44 | 54.33  | 26225.77 | 0.34 |
| 2025 | 26925.19 | 27017.62 | 26197.64 | 49.62  | 26247.26 | 0.34 |

In [ ]:
geometrisk_sum = dataset.aggregate_array('GEOMETRISK').reduce(ee.Reducer.sum()).getInfo()
geometrisk_sum_km = ee.Number(geometrisk_sum).divide(pow(10,2)).getInfo()
print(f"Sum of total area of all field blocks:", geometrisk_sum_km )

area_sum = dataset.aggregate_array('GB_AREAL').reduce(ee.Reducer.sum()).getInfo()
area_sum_km = ee.Number(area_sum).divide(pow(10,2)).getInfo()
print(f"Sum of subsidized area of all field blocks:", area_sum_km )

non_subsidized_sum = dataset.aggregate_array('GB_FRADRAG').reduce(ee.Reducer.sum()).getInfo()
non_subsidized_sum_km = ee.Number(non_subsidized_sum).divide(pow(10,2)).getInfo()
print(f"Sum of non-subsidized area of all field blocks:", non_subsidized_sum_km )

In [ ]:
def add_area(feature):
    return feature.set('area_m2', feature.geometry().area())

# Add column with feature area
with_area = dataset.map(add_area)
total_area = with_area.aggregate_sum('area_m2').getInfo()
field_area_km = ee.Number(total_area / pow(10,6))
print("Total area som coming from the polygons (km²):" )
display(field_area_km)

## Merging with Dynamic World

Loading DW Image with the year mode of Denmark and removing non DAA areas from DW image.

In [28]:
dw_image = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR) + '_mode').clipToCollection(DENMARK)
display(dw_image)

clipped_img = dw_image.clipToCollection(dataset)

## Out of the entirety of Denmark, how much did DW predict as agriculture?

|  | DW Crop Area | Area Outside DAA | Percentage of Crop Classifications Outside DAA |
| --- | --- | --- | --- | 
| 2020 | 23950.46 | 1845.86 | 7.71 |
| 2021 | 22870.00 | 1847.04 | 8.08 |
| 2022 | 20876.43 | 1620.94 | 7.76 |
| 2023 | 23664.82 | 2122.89 | 8.97 |
| 2024 | 22890.48 | 1888.50 | 8.25 |
| 2025 | 20133.85 | 1544.56 | 7.67 |

Calculating the area for each Dynamic World classification type (Code adopted from [here](https://spatialthoughts.com/2020/06/19/calculating-area-gee/))


In [33]:
DENMARK.geometry().area()

In [7]:
def area_class(dw_class):
    area_image = dw_class.multiply(ee.Image.pixelArea())
    area = area_image.reduceRegion( 
        reducer = ee.Reducer.sum(),
        scale = 10,
        maxPixels = 1e13,
    )
    return ee.Number(area.get('label')).divide(pow(10,6))

In [29]:
dw_crops = dw_image.eq(4)
dw_crops_area = area_class(dw_crops)
print("DW Crops Area:")
display(dw_crops_area)

DW Crops Area:


## How much of the area inside of DAA got classified as each DW class?

|  | Water | Trees | Grass | Flooded Vegetation | Crops | Shrub & Scrub | Built | Bare | Snow & Ice |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2020 | 66.88 | 2299.63 | 2859.69 | 57.19 | 22104.60 | 124.57 | 315.67 |  8.00 | 0.55 |
| 2021 | 61.45 | 2355.34 | 3747.81 | 41.42 | 21022.96 |  72.76 | 285.91 |  8.11 | 29.88| 
| 2022 | 73.25 | 1944.87 | 5651.77 | 44.16 | 19255.49 | 147.29 | 288.41 | 12.10 | 0.84 |
| 2023 | 64.95 | 2098.40 | 2908.20 | 51.30 | 21541.93 | 115.59 | 287.00 | 13.88 | 0.68 |
| 2024 | 59.55 | 2875.20 | 2748.15 | 49.60 | 21001.98 |  59.27 | 240.01 | 12.11 | 1.25 |
| 2025 | 64.06 | 2440.08 | 5470.97 | 27.95 | 18589.29 | 154.01 | 222.01 | 22.65 | 0.89 |

### Solar panels

Only available from 2024

|  | Water | Trees | Grass | Flooded Vegetation | Crops | Shrub & Scrub | Built | Bare | Snow & Ice |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | 0.19 | 2.08 | 2.76 | 0.13 | 1.42 | 9.66 | 1.48 |  7.79 | 0.28 |
| 2024 | 0.02 | 1.41 | 0.98 | 0.13 | 1.02 | 5.26 | 0.60 |  2.48 | 0.15 |

### Greenhouses
|  | Water | Trees | Grass | Flooded Vegetation | Crops | Shrub & Scrub | Built | Bare | Snow & Ice |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | 0.00 | 0.03 | 0.02 | 0.00 | 0.15 | 0.00 | 0.57 | 0.01 | 0.06 |
| 2024 | 0.00 | 0.02 | 0.02 | 0.00 | 0.22 | 0.00 | 0.60 | 0.00 | 0.00

### Permanent Grass
|  | Water | Trees | Grass | Flooded Vegetation | Crops | Shrub & Scrub | Built | Bare | Snow & Ice |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | 31.14 | 868.27 | 1376.87 | 18.85 | 297.84 | 82.92 | 53.43 | 2.25 | 0.17 |

In [ ]:
# Calculating how much of the DAA got classified as each class
for i in range(9):
    dw_class = clipped_img.eq(i)
    daa_area = area_class(dw_class)
    print(f"DAA Area Classified as", i)
    display(daa_area)

## Change Comparisons: Was change detected by DW? Was change detected by DAA?
### How much change happened between two DAA datasets?

|  | Overall Change | New Field Area | Removed Field Area | Total Area | DAA & DW Agree | Agree Percentage | DW Change (inside DAA) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2024-2025 |  -57.63 | 90.54 | 148.18 | 238.72 | 24.55 | 10.29 | 4556.40 |
| 2023-2024 |  -36.58 | 81.43 | 118.01 | 199.44 | 21.45 | 10.76 | 3340.34 |
| 2022-2023 | -337.32 | 93.44 | 430.77 | 524.21 | 26.08 | 4.98 | 4575.79 |
| 2021-2022 | -206.47 | 52.94 | 259.41 | 312.36 | 13.08 | 4.19 | 4764.25 |
| 2020-2021 | -207.62 | 79.83 | 287.46 | 367.29 | 14.95 | 4.07 | 3583.38 |


In [ ]:
dataset_after = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR))
dataset_before = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR - 1))

# Transform data to raster format for lighter operations
after_img = ee.Image.constant(0).paint(dataset_after, 1)
before_img = ee.Image.constant(0).paint(dataset_before, 1)

# Image with are that changed from DAA (1 - new field, -1 - removed field)
change_img = after_img.subtract(before_img)
# Removing places where there was no change
change_img = change_img.updateMask(change_img.neq(0))

In [ ]:
def area_contrast_img(change_img):
    change_area = change_img.multiply(ee.Image.pixelArea())
    change_area = change_area.reduceRegion( 
        reducer = ee.Reducer.sum(),
        scale = 10,
        maxPixels = 1e13,
        geometry = DENMARK
    )
    return ee.Number(change_area.get('constant')).divide(pow(10,6))   

positive_change_area = area_contrast_img(change_img.eq(1))
print("Amount of new field area: ")
display(positive_change_area)

negative_change_area = area_contrast_img(change_img.eq(-1))
print("Amount of field area which disappeared: ")
display(negative_change_area)

overall_change_area = positive_change_area.subtract(negative_change_area)
print("Overall amount of changed field area: ")
display(overall_change_area)

total_change_area = positive_change_area.add(negative_change_area)
print("Total amount of changed field area: ")
display(total_change_area)

In [ ]:
change_map = geemap.Map(scroll_wheel_zoom = False)

change_map.add_layer(
    change_img,
    {'min': -1, 'max': 1, 'palette': ['red', 'white', 'green']},
    'DAA Changed Area',
)
change_map

### How much change did DW detect inside the field area? How much of it was coherent with DAA data?

In [ ]:
dw_image_before = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20'+ str(YEAR) + '_mode')
dw_image_after = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20'+ str(YEAR - 1) + '_mode')

# dw detected change image
dw_change_img = dw_image_after.neq(dw_image_before)
dw_change_img = dw_change_img.updateMask(dw_change_img.neq(0))

# How much change DW detected inside of DAA defined area
dw_detected_change_img = dw_change_img.clipToCollection(dataset_after)
# intersection dw detected change area with DAA defined change area
dw_agree_change_img = dw_change_img.updateMask(change_img)

In [ ]:
agree_change_area = area_class(dw_agree_change_img)
print("DAA and Dynamic World agreed area where change in fields happened: ")
display(agree_change_area)

detected_change_area = area_class(dw_detected_change_img)
print("Dynamic World detected this much change inside of the DAA defined area: ")
display(detected_change_area)

In [ ]:
dw_change_map = geemap.Map(scroll_wheel_zoom = False)

dw_change_map.add_layer(
    dw_agree_change_img,
    {'min': -1, 'max': 1, 'palette': ['red', 'white', 'green']},
    'DAA Changed Area',
)
dw_change_map

## Visual Representation of DW Intersection with Markblokke

In [ ]:
# img = linked_col.mosaic
overlap_map = geemap.Map(scroll_wheel_zoom = False, basemap='ROADMAP')
# map.centerObject(s2_img, 12)

# Add DW classification layer
overlap_map.add_layer(
    dw_image,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
# Add DAA polygon area layer
overlap_map.add_layer(
    dataset, 
    {'color': 'white', 'oppacity': 0.2},
    "Polygon Boundaries",
)
overlap_map

In [ ]:
map = geemap.Map(scroll_wheel_zoom = False, basemap='ROADMAP')

# map.add_layer(
#     s2_img,
#     S2_VIZ_PARAMS, 
#     'Sentinel-2 L1C',
#     False
# )
map.add_layer(
    clipped_img,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map